# Assignment 03 solutions

---

## Problem 1: Dynamic Array Accessors

See the assignment notebook for the full problem statement.

**Solution: technical note**

All seven methods are small, and six of them are one idea each. The interesting ones are the last three.

`__len__`, `get_size`, and `get_capacity` are straight reads of state the class already tracks. The only thing to get right is *which* number: `_size` counts values, `_capacity` counts slots. They differ whenever the array is not exactly full, so returning `_capacity` from `__len__` would be wrong -- `len()` means "how many items are in this collection".

`get` needs an explicit bounds check. `0 <= index < self._size` is the whole contract: it rejects negatives, it rejects positions at or past `_size`, and it does so *before* touching `_underlying`. Leaving the check out and indexing directly would still pass every positive-index test, then quietly return the last element for `get(-1)`, because Python lists accept negative indices natively. Writing the guard first, not the lookup first, is what makes that case a non-event.

`index_of` is the linear search from class, with one extra requirement: stop at `_size`, not at `len(self._underlying)`. The slots past `_size` all hold `-1`, so a search that ran to the end of the underlying list would report a sentinel slot for `index_of(-1)`. Since the Pact rules out `break`, the loop carries its own exit condition -- `found_at == self._EMPTY` -- which stops the scan as soon as there is an answer and doubles as the not-found return value.

`contains` is deliberately a one-liner over `index_of`. This is the refactor from class: `contains` and `index_of` were the same loop with different return values, so the honest version keeps one loop and derives the other answer from it. Writing a second loop here is not wrong in its output -- it is duplicated logic, which is why the constraint is checked on the source rather than the result.

`remove` reuses `index_of` for the same reason, then does the only real work in the problem: closing the gap. Everything after the removed position shifts one slot left, which means walking **forward** from `position` and copying each neighbour back -- walking backward would overwrite values before copying them. Three things have to be true when the shift finishes: the vacated slot at the end holds `_EMPTY` again, `_size` has dropped by one, and `_capacity` is untouched. Forgetting the first leaves a stale duplicate of the last value sitting in the array, which nothing in the printed output would reveal, because `__str__` only ever looks at the first `_size` slots.

---

## Problem 2: Dynamic Array Magic Methods

See the assignment notebook for the full problem statement.

**Solution: technical note**

`from_values` exists so that `__repr__` has something to name. It is a `@classmethod` because it builds an instance rather than using one, and it builds through `cls()` rather than `DynamicArray()`. That single choice is what makes the method inheritable: called as `BoundedArray.from_values([...])`, `cls` *is* `BoundedArray`, so the result is a `BoundedArray`. Hardcoding the class name would silently hand back the wrong type to every subclass, and nothing in this problem's own output would reveal it.

`__eq__` compares `type(self)` with `type(other)`, not `isinstance(other, DynamicArray)`. The two agree on everything except subclasses, and that is exactly the case we care about: a `BoundedArray` holding `[1, 2]` is not interchangeable with a plain `DynamicArray` holding `[1, 2]`, because it carries a different promise about what it will accept next. The comparison also short-circuits, which is what keeps `a == None` and `a == "text"` from ever reaching `other._size` and raising.

Capacity stays out of the comparison on purpose. Two arrays holding the same three values are the same array as far as a caller is concerned; that one of them has room for sixty-four values is a fact about the implementation, not about the data. Deciding *which attributes decide sameness* is the real content of writing an `__eq__`, and it is a judgement call rather than a mechanical one.

The loop carries `result` in its condition (`while i < self._size and result`) so it stops at the first mismatch without a `break`. The same shape as `index_of`, for the same reason.

`__repr__` builds its name from `type(self).__name__`. The aim of a `__repr__` is a string that could rebuild the object, so pasting its output back into Python has to produce an equal array -- and for a subclass it has to rebuild *that* subclass, not the parent. This is the payoff for `from_values` using `cls`: the two decisions are the same decision, seen from opposite ends.

`__add__` returns a brand-new array and touches neither operand, matching the rule arithmetic already follows -- `x + y` does not modify `x`. The common bug is to `add` to `self` and return it, which passes a naive "is the result right?" test and corrupts the left operand for every later line. It returns a plain `DynamicArray` rather than `type(self)()` because a concatenation of two subclass instances can easily be longer than that subclass permits.


---

## Problem 3: Cards, Properties and Decorators

See the assignment notebook for the full problem statement.

**Solution: technical note**

Four of the five `Card` members are one idea each; the decorator is the one worth slowing down for.

`is_valid_rank` is a `@staticmethod` because the question it answers — is this string a legal rank? — is about a rank, not about any particular card. It reads no `self` and no `cls`, so it sits inside `Card` by topic rather than by data, and it can be called on the class or on an instance. Membership with `in` is fine here: `_RANKS` is a fixed 13-element constant, not the linear search Problem 1 exists to drill.

`rank` is two methods sharing one name, and the sharing is the point. `@property` above the first rebinds `rank` to a property object — the same rule as any decorator, which is why `type(Card.rank)` is `property` while `type(Card.label)` is `function`. `@rank.setter` above the second asks that property for a copy of itself carrying this setter, and binds *that* back to `rank` again. Rename the second method `set_rank` and nothing is rebound: the read-only property survives untouched with an unrelated method beside it, and the assignment `card.rank = "K"` fails just as it did before.

The setter is where validation lives, and it reuses `is_valid_rank` rather than repeating the list. A plain attribute would have accepted `"1"` without comment; that is the whole argument for paying the cost of a property. Note what `__init__` does *not* do: it assigns `self._rank` directly, so `Card("1", "hearts")` is built without complaint. That is the lecture's `Account(-50)` warning in this file — a setter guards assignment through the property, and `__init__` goes around it. Here it is also deliberate, so that a `Card` can still be constructed while the setter is unfinished.

`color` is computed on every read rather than stored in `__init__`, and it has no setter. Both halves matter. Storing it would create a second copy of a fact already recorded by `_suit`, free to drift; omitting the setter makes assignment raise `AttributeError`, which is the honest answer, since a card's color is not an independent thing you can set. Store what you are given, compute what you can derive.

`__repr__` uses `f"Card({self._rank!r}, {self._suit!r})"`. The `!r` is the whole difference: it asks each value for its own `repr`, which supplies the quotes, so the output is `Card('7', 'hearts')` — code that rebuilds an equal card. Plain concatenation yields `Card(7, hearts)`, which looks nearly identical and rebuilds nothing, because `7` and `hearts` are not valid expressions there. The handout's version has exactly this gap; it is fixed here.

`announce` returns a different function than it was given. `wrapper` closes over `func`, so each decorated function gets its own wrapper remembering its own target, and `*args` lets one decorator serve functions of any arity. `@announce` above `def describe` means `describe = announce(describe)`, which is why `describe.__name__` afterwards is `'wrapper'` — the name really does point somewhere else now. The definition lives inside the main guard on purpose: a decorator applied at module level would run the moment the file is imported, so an unfinished `announce` would take the whole file down with it, including the problems that have nothing to do with it.

---

## Problem 4: Bounded Arrays, Inheritance and Name Mangling

See the assignment notebook for the full problem statement.

**Solution: technical note**

Three small methods, and almost all of the content is in what they *do not* do.

`__init__` calls `super().__init__()` and stops. The temptation is to set `_underlying`, `_capacity` and `_size` here as well, since they are right there in the parent and easy to copy. That version works today and rots tomorrow: the moment `DynamicArray.__init__` changes, the child quietly stops matching it. Calling up is not politeness, it is the thing that keeps one definition of what a `DynamicArray` is.

The parameter has a default, and that is load-bearing rather than convenient. `from_values`, written back in Problem 2, builds its result with `cls()` and passes nothing. Give `limit` no default and `BoundedArray.from_values([...])` fails with a `TypeError` about a missing argument — a method written before this class existed, broken by a signature choice made here. Inheritance means code you did not write calls code you did.

`add` is the only override, and it delegates. Check the limit, then `super().add(value)`. Writing to `self._underlying[self._size]` directly would duplicate the parent's resize-then-store logic, and would silently skip the parent's own guard against negative values. The child's job is to add a rule, not to reimplement the work.

The payoff shows up in three inherited behaviors that nobody wrote for `BoundedArray`. `from_values` returns a `BoundedArray`, because it builds with `cls()`. It enforces the limit, because it calls `add`, which is now this one. And `__repr__` prints `BoundedArray.from_values([...])`, because it builds its name from `type(self).__name__`. Each of those was a small decision made in an earlier problem, and each pays off only now.

`self.__limit` uses two underscores, and the rewriting is the lesson. Inside a class body Python turns `__limit` into `_BoundedArray__limit` at compile time, choosing the name from the class the line is *written in* — not the class of the object at runtime. That is why `vars(b)` lists `_BoundedArray__limit` right beside `_capacity`, `_size` and `_underlying`, all equally visible; why `b.__limit` raises `AttributeError` from outside, since nothing is stored under that spelling; and why `b._BoundedArray__limit` returns 8. Nothing was hidden. The attribute was filed under a different name, and that is the whole mechanism.

So mangling is not privacy, and it never was. What it actually prevents is a *subclass* writing `self.__limit` for some unrelated purpose and silently clobbering this one — each class gets its own key. One underscore protects a caller from a detail; two protect a class from a subclass. Ask which you are doing, and let the answer pick the spelling.

One last distinction worth keeping straight, and the example is now built to show it: after six values, `len(b)` is 6 while `get_capacity()` is 8. The two numbers differ on purpose. Capacity is how many slots the underlying array has, and it doubles on demand; the limit is a promise about how many values this object will ever accept. An unbounded `DynamicArray` grows forever; a `BoundedArray` stops, and raises `IndexError` rather than failing quietly, because losing a value with no warning is worse than an exception that says exactly what went wrong.

All four problems share one file. `DynamicArray` carries Problems 1 and 2, `Card` and `announce` are Problem 3, and `BoundedArray` is Problem 4; comment banners mark which problem each group of methods belongs to.

In [ ]:
class DynamicArray:
    """A fixed-capacity array of non-negative integers that doubles its
    capacity whenever it fills up.
    """

    # Class-level constants: shared by every instance, not stored per object.
    # Naming them documents the policy and keeps the literals out of the code.
    _DEFAULT_CAPACITY: int = 4
    _GROWTH_FACTOR: int = 2
    _EMPTY: int = -1  # sentinel: "this slot holds no value"

    def __init__(self, capacity: int = _DEFAULT_CAPACITY) -> None:
        # A capacity below 1 would leave nowhere to put the first value.
        if capacity < 1:
            raise ValueError("capacity must be at least 1")
        self._capacity: int = capacity
        self._size: int = 0
        # Pre-fill every slot with the sentinel, so the underlying list
        # always holds exactly _capacity entries.
        self._underlying: list[int] = []
        for _ in range(self._capacity):
            self._underlying.append(self._EMPTY)

    def __str__(self) -> str:
        # Show only the filled slots -- sentinel slots are internal detail.
        inside = ""
        for i in range(self._size):
            inside = inside + str(self._underlying[i])
            if i < self._size - 1:
                inside = inside + ", "
        return "[" + inside + "]"

    def resize(self) -> None:
        # Build a bigger array, copy what is real, then swap it in.
        bigger_capacity = self._capacity * self._GROWTH_FACTOR
        temp: list[int] = []
        for _ in range(bigger_capacity):
            temp.append(self._EMPTY)
        # Only the first _size entries hold real values; the rest are
        # already sentinels in temp.
        for i in range(self._size):
            temp[i] = self._underlying[i]
        self._underlying = temp
        self._capacity = bigger_capacity

    def add(self, value: int) -> None:
        # A negative value could not be told apart from the sentinel.
        if value < 0:
            raise ValueError("values must be non-negative")
        # Grow first if the array is full, so index _size is always writable.
        if self._size >= self._capacity:
            self.resize()
        # _size is both the count of values and the index of the next slot.
        self._underlying[self._size] = value
        self._size = self._size + 1

    # ----- Problem 1: accessors -----

    def __len__(self) -> int:
        # The number of values stored, not the number of slots.
        return self._size

    def get_size(self) -> int:
        # Same value as __len__, under a name that needs no dunder knowledge.
        return self._size

    def get_capacity(self) -> int:
        # Total slots, including the empty sentinel ones.
        return self._capacity

    def get(self, index: int) -> int:
        # Anything outside the filled range reports the sentinel rather than
        # raising -- including negative indices, which a plain Python list
        # would happily accept as counting from the end.
        result = self._EMPTY
        if 0 <= index < self._size:
            result = self._underlying[index]
        return result

    def index_of(self, value: int) -> int:
        # Linear search over the filled slots only. Searching the sentinel
        # slots as well would make index_of(-1) find one of them.
        found_at = self._EMPTY
        i = 0
        while i < self._size and found_at == self._EMPTY:
            if self._underlying[i] == value:
                found_at = i
            i = i + 1
        return found_at

    def contains(self, value: int) -> bool:
        # Reuses index_of rather than repeating the search loop.
        return self.index_of(value) != self._EMPTY

    def remove(self, value: int) -> bool:
        # Find the value first, then close the gap it leaves behind.
        position = self.index_of(value)
        removed = position != self._EMPTY
        if removed:
            # Shift every later value one slot to the left.
            i = position
            while i < self._size - 1:
                self._underlying[i] = self._underlying[i + 1]
                i = i + 1
            # The last occupied slot is now a duplicate of its neighbour.
            self._underlying[self._size - 1] = self._EMPTY
            self._size = self._size - 1
        return removed

    # ----- Problem 2: magic methods -----

    @classmethod
    def from_values(cls, values: list[int]) -> "DynamicArray":
        # Building through cls() rather than DynamicArray() means a subclass
        # calling this gets an instance of itself, not of the parent.
        built = cls()
        for value in values:
            built.add(value)
        return built

    def __eq__(self, other: object) -> bool:
        # Same concrete type and same values in the same order. Capacity is
        # an implementation detail, so two arrays holding the same values
        # are equal even when their arrays are different sizes.
        # Comparing type(self) against type(other), rather than asking
        # isinstance, is what stops a subclass from being equal to a plain
        # DynamicArray holding the same values. It also short-circuits, so
        # comparing against a string or None never reaches ._size.
        result = type(self) is type(other) and self._size == other._size
        if result:
            i = 0
            while i < self._size and result:
                result = self._underlying[i] == other._underlying[i]
                i = i + 1
        return result

    def __repr__(self) -> str:
        # Aim for a string that could rebuild the object. type(self).__name__
        # rather than a hardcoded "DynamicArray" so a subclass reports itself.
        inside = ""
        for i in range(self._size):
            inside = inside + str(self._underlying[i])
            if i < self._size - 1:
                inside = inside + ", "
        return type(self).__name__ + ".from_values([" + inside + "])"

    def __add__(self, other: "DynamicArray") -> "DynamicArray":
        # Builds and returns a NEW array; neither operand is modified.
        # The result is always a plain DynamicArray, because a concatenation
        # can be longer than a subclass would allow its own instances to be.
        combined = DynamicArray()
        i = 0
        while i < self._size:
            combined.add(self._underlying[i])
            i = i + 1
        j = 0
        while j < other._size:
            combined.add(other._underlying[j])
            j = j + 1
        return combined


class Card:
    """One playing card: a rank and a suit that travel together."""

    # The 13 legal ranks, in order. Naming them keeps the check in one place.
    _RANKS: tuple[str, ...] = (
        "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K", "A",
    )
    # The two red suits; every other legal suit is black.
    _RED_SUITS: tuple[str, ...] = ("hearts", "diamonds")

    def __init__(self, rank: str, suit: str) -> None:
        # Assigned directly rather than through the rank property, so that a
        # Card can still be built while the setter is unfinished. Note the
        # consequence, which is the lecture's warning about Account(-50):
        # __init__ bypasses the setter, so it performs no validation.
        self._rank: str = rank
        self._suit: str = suit

    def label(self) -> str:
        # A plain method, not a property: type(Card.label) is function,
        # while type(Card.rank) is property.
        return f"{self._rank} of {self._suit}"


    def __eq__(self, other: object) -> bool:
        # Same concrete type, same rank, same suit.
        return (
            type(self) is type(other)
            and self._rank == other._rank
            and self._suit == other._suit
        )

    @staticmethod
    def is_valid_rank(rank: str) -> bool:
        # Neither self nor cls is needed: this asks about a rank, not about
        # any particular card, so it sits with Card by topic rather than by
        # data. Callable as Card.is_valid_rank(...) or on an instance.
        return rank in Card._RANKS

    @property
    def rank(self) -> str:
        # Reading a_card.rank runs this method, with no parentheses at the
        # call site -- the caller keeps plain attribute syntax and the class
        # keeps control of the value.
        return self._rank

    @rank.setter
    def rank(self, value: str) -> None:
        # The setter is where the check lives. A plain attribute would accept
        # anything; assignment now runs this instead.
        if not Card.is_valid_rank(value):
            raise ValueError(f"{value!r} is not one of the 13 legal ranks")
        self._rank = value

    @property
    def color(self) -> str:
        # Computed, and read-only because there is no matching setter:
        # the color follows from the suit, so storing it separately would
        # only create something that could go stale.
        result = "black"
        if self._suit in Card._RED_SUITS:
            result = "red"
        return result

    def __repr__(self) -> str:
        # !r asks for each value's own repr, which supplies the quotes, so
        # pasting this output back into Python rebuilds an equal card.
        return f"Card({self._rank!r}, {self._suit!r})"


def announce(func):
    """Wrap `func` so every call announces itself before running."""
    # A decorator takes the function defined below it and returns the object
    # the name should point at instead -- here, wrapper.
    def wrapper(*args):
        # *args accepts whatever the wrapped function takes, so one decorator
        # works for functions of any arity.
        print("calling " + func.__name__)
        return func(*args)
    return wrapper


class BoundedArray(DynamicArray):
    """A DynamicArray that refuses to hold more than a fixed number of
    values, however much room the underlying array happens to have.
    """

    # The most values a BoundedArray accepts unless told otherwise.
    _DEFAULT_LIMIT: int = 6

    def __init__(self, limit: int = _DEFAULT_LIMIT) -> None:
        # super() runs DynamicArray's initializer, so the underlying array,
        # the capacity and the size are set up exactly as the parent does
        # it. Takes no required argument, so inherited code that builds a
        # new instance with cls() keeps working.
        super().__init__()
        # Two leading underscores: the compiler rewrites this name to
        # _BoundedArray__limit, using the class the line is written in.
        self.__limit: int = limit

    def get_limit(self) -> int:
        # Inside the class the short name works, because the compiler
        # rewrites it here in exactly the same way.
        return self.__limit

    def add(self, value: int) -> None:
        # Overrides DynamicArray.add. Check the limit, then hand the actual
        # storing back to the parent rather than repeating its logic.
        if self._size >= self.__limit:
            raise IndexError("array is full")
        super().add(value)

**Test cases**

In [ ]:
# ----- Problem 1 -----
da = DynamicArray()
da.add(10001)
da.add(60626)
da.add(90210)
print(da)                     # expected: [10001, 60626, 90210]
print(len(da), da.get_size(), da.get_capacity())   # expected: 3 3 4

print(da.get(0), da.get(2))   # expected: 10001 90210
print(da.get(-1), da.get(3), da.get(100))          # expected: -1 -1 -1

print(da.index_of(10001), da.index_of(90210), da.index_of(99999))
# expected: 0 2 -1
print(da.contains(60626), da.contains(99999))      # expected: True False

da.add(11111)
da.add(22222)
print(da)                     # expected: [10001, 60626, 90210, 11111, 22222]
print(len(da), da.get_capacity())                  # expected: 5 8

print(da.remove(90210))       # expected: True
print(da)                     # expected: [10001, 60626, 11111, 22222]
print(len(da), da.get_capacity())                  # expected: 4 8
print(da.remove(90210))       # expected: False
print(da.index_of(22222))     # expected: 3

# ----- Problem 2 -----
a = DynamicArray.from_values([10001, 60626, 90210])
print(a)                          # expected: [10001, 60626, 90210]
print(len(a), a.get_capacity())   # expected: 3 4

print(repr(a))
# expected: DynamicArray.from_values([10001, 60626, 90210])
print(repr(DynamicArray()))       # expected: DynamicArray.from_values([])

b = DynamicArray.from_values([10001, 60626, 90210])
print(a == b)                     # expected: True
print(a is b)                     # expected: False

# Same values, very different capacity -- still equal.
wide = DynamicArray(64)
wide.add(10001)
wide.add(60626)
wide.add(90210)
print(a == wide, a.get_capacity(), wide.get_capacity())   # expected: True 4 64

print(a == DynamicArray.from_values([10001, 60626]))         # expected: False
print(a == DynamicArray.from_values([90210, 60626, 10001]))  # expected: False
print(a == "not an array")        # expected: False
print(a == None)                  # expected: False

c = a + DynamicArray.from_values([11111, 22222])
print(c)                          # expected: [10001, 60626, 90210, 11111, 22222]
print(len(c))                     # expected: 5
print(a)                          # expected: [10001, 60626, 90210]  (unchanged)
print(len(a))                     # expected: 3  (unchanged)

empty = DynamicArray()
print(empty + empty)              # expected: []
print(a + empty == a)             # expected: True

# ----- Problem 3 -----
card = Card("7", "hearts")

print(card.label())                    # expected: 7 of hearts
print(card.rank)                       # expected: 7
print(card.color)                      # expected: red
print(Card("K", "spades").color)       # expected: black

print(Card.is_valid_rank("Q"))         # expected: True
print(Card.is_valid_rank("1"))         # expected: False
print(card.is_valid_rank("10"))        # expected: True

print(repr(card))                      # expected: Card('7', 'hearts')
print([card, card])
# expected: [Card('7', 'hearts'), Card('7', 'hearts')]

# The text of the repr is the code that rebuilds the card: typing it back
# in by hand produces an equal Card. Shown this way rather than with
# eval(), which runs whatever string it is handed and has no place in
# ordinary code.
print(Card("7", "hearts") == card)     # expected: True

# The setter validates; the getter just reads.
card.rank = "K"
print(card.rank, card.label())         # expected: K K of hearts

try:
    card.rank = "1"
except ValueError as error:
    print("ValueError:", error)
    # expected: ValueError: '1' is not one of the 13 legal ranks

# color has a getter and no setter, so it is read-only. Assigning to it
# on purpose here, to show what that means. ty is right to object, so the
# line carries ty's own ignore directive. Use that spelling: mypy's
# equivalent comment is written differently, and ty reads it as a
# malformed directive and warns about it.
try:
    card.color = "green"  # ty: ignore[invalid-assignment]
except AttributeError:
    print("AttributeError as expected")
    # expected: AttributeError as expected


# ----- Problem 4 -----
bounded = BoundedArray()
print(type(bounded).__name__, bounded.get_limit())   # expected: BoundedArray 6

for value in range(6):
    bounded.add(value)
print(len(bounded), bounded.get_capacity())          # expected: 6 8

# The override refuses the seventh value; the parent would have resized.
# Capacity is 8 here and the limit is 6: room in the array is not
# permission to use it.
try:
    bounded.add(99)
except IndexError as error:
    print("IndexError:", error)
    # expected: IndexError: array is full

# Inherited from Problem 2, and still returning the right class, because
# from_values builds with cls() rather than with DynamicArray().
pair = BoundedArray.from_values([1, 2])
print(type(pair).__name__, pair)                     # expected: BoundedArray [1, 2]
print(repr(pair))                                    # expected: BoundedArray.from_values([1, 2])

# from_values calls add, so the subclass's limit is enforced through it.
try:
    BoundedArray.from_values([0, 1, 2, 3, 4, 5, 6])
except IndexError as error:
    print("IndexError:", error)
    # expected: IndexError: array is full

# Same values, different class: not equal.
print(pair == DynamicArray.from_values([1, 2]))      # expected: False
print(pair == BoundedArray.from_values([1, 2]))      # expected: True

# Name mangling, in the open. Nothing is hidden -- the attribute is
# simply filed under a rewritten key.
print(sorted(vars(bounded)))
# expected: ['_BoundedArray__limit', '_capacity', '_size', '_underlying']

# ----- Problem 3, continued: applying the decorator -----
#
# Kept last on purpose. Decorating describe adds a printed line, so if
# this block sat higher up, forgetting the decorator would shift every
# line below it and make the other problems look wrong too.
# A decorator rebinds the name to the wrapper. Defined here, inside the
# main guard, so that importing this file never runs it.
@announce
def describe(rank: str, suit: str) -> str:
    return f"{rank} of {suit}"

print(describe("7", "hearts"))
# expected: calling describe
# expected: 7 of hearts
print(describe.__name__)               # expected: wrapper